# WaterMelonBraker: YOLOv8s fine-tune for watermelon detection (Kaggle版)

Roboflow の vietnam-fruit-in-lab/watermelon-n4rrw データセットを YOLOv8s で fine-tune し、Quest 3 の Sentis 用に ONNX エクスポートするノートブック。

**ランタイム**: Kaggle T4 x2 GPU（無料枠）で 20〜40分

## 事前準備

1. Notebook 右サイドバーの `Settings` から `Accelerator` を `GPU T4 x2` or `GPU P100` に変更
2. 電話番号認証済みであること（GPU使用の必須条件）
3. Roboflow API Key を Kaggle Secrets に登録
   - 右サイドバー `Add-ons` → `Secrets`
   - `+ Add a new secret` で `ROBOFLOW_API_KEY` として登録
   - `Attached` トグルを ON


## Step 1: 環境確認

In [ ]:
!nvidia-smi

## Step 2: 必要なパッケージをインストール

In [ ]:
!pip install -q ultralytics roboflow onnx onnxsim

## Step 3: Roboflow API Key を取得(Kaggle Secrets経由)

In [ ]:
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
ROBOFLOW_API_KEY = user_secrets.get_secret('ROBOFLOW_API_KEY')
print('API Key loaded:', ROBOFLOW_API_KEY[:8] + '...' if ROBOFLOW_API_KEY else 'FAILED')

## Step 4: データセットのダウンロード

vietnam-fruit-in-lab の watermelon-n4rrw を使用。

In [ ]:
import os
os.chdir('/kaggle/working')

from roboflow import Roboflow

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace('vietnam-fruit-in-lab').project('watermelon-n4rrw')
version = project.version(1)
dataset = version.download('yolov8')

print('Dataset location:', dataset.location)
!ls {dataset.location}

## Step 5: data.yaml を確認

In [ ]:
!cat {dataset.location}/data.yaml

## Step 6: YOLOv8s を fine-tune

**設定**

- `model='yolov8s.pt'`: small (11M params)、遠距離検出に適する
- `epochs=100`: 106枚と少ないデータなので多めに
- `imgsz=640`: 標準
- `batch=16`: T4 GPU (16GB)で余裕
- `patience=20`: 20エポック改善なければ早期終了
- `augment=True`: augmentation強化(少ないデータで過学習を避ける)

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8s.pt')

results = model.train(
    data=f'{dataset.location}/data.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,
    project='/kaggle/working/runs/train',
    name='watermelon',
    exist_ok=True,
    augment=True,
)

## Step 7: 学習結果の可視化

In [ ]:
from IPython.display import Image, display

display(Image('/kaggle/working/runs/train/watermelon/results.png'))
display(Image('/kaggle/working/runs/train/watermelon/confusion_matrix.png'))

## Step 8: 検証セットで動作確認

In [ ]:
best_model = YOLO('/kaggle/working/runs/train/watermelon/weights/best.pt')
metrics = best_model.val(data=f'{dataset.location}/data.yaml')
print(f'mAP@50:    {metrics.box.map50:.4f}')
print(f'mAP@50-95: {metrics.box.map:.4f}')

## Step 9: ONNX 形式にエクスポート

Meta 公式サンプルの `SentisModelEditorConverter.cs` が期待する出力形状 (1, 84, N) の標準 YOLOv8 ONNX。

In [ ]:
onnx_path = best_model.export(
    format='onnx',
    imgsz=640,
    opset=15,
    simplify=True,
    dynamic=False,
    nms=False,
)
print('ONNX saved to:', onnx_path)

## Step 10: クラス名リストを出力

In [ ]:
class_names = best_model.names
print('Class count:', len(class_names))

with open('/kaggle/working/SentisYoloClasses.txt', 'w') as f:
    for i in range(len(class_names)):
        f.write(class_names[i] + '\n')

print('\n=== SentisYoloClasses.txt ===')
with open('/kaggle/working/SentisYoloClasses.txt') as f:
    print(f.read())

watermelon_id = [i for i, name in class_names.items() if 'watermelon' in name.lower()]
print(f'watermelon class ID: {watermelon_id}')

## Step 11: 成果物をKaggle出力ディレクトリに整理

Kaggle Notebook 完了後、右サイドバーの `Output` からダウンロード可能。

In [ ]:
import shutil

shutil.copy(onnx_path, '/kaggle/working/watermelon.onnx')
shutil.copy('/kaggle/working/runs/train/watermelon/weights/best.pt', '/kaggle/working/watermelon_best.pt')

print('Files ready in /kaggle/working/:')
!ls -la /kaggle/working/*.onnx /kaggle/working/*.pt /kaggle/working/SentisYoloClasses.txt